# Endosomal Biomarker ML Pipeline
**Simoes et al. 2020 | VPS35-cKO Mouse CSF Proteomics | Stability Selection + SHAP**

This notebook runs the full pipeline end-to-end.
All functions are defined inline for Colab compatibility.

In [ ]:
# ── Cell 1: Install dependencies ──
import subprocess, sys
pkgs = ['xgboost>=2.0.0', 'shap>=0.42.0', 'openpyxl>=3.1.0', 'tqdm>=4.65.0']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies installed.')


In [ ]:
# ── Cell 2: Imports ──
import os, json, time, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from io import StringIO
warnings.filterwarnings('ignore')

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
ROOT = Path('/content/endosomal-biomarker-ml') if IN_COLAB else Path('..') 
DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROC = ROOT / 'data' / 'processed'
FIGURES = ROOT / 'figures'
RESULTS = ROOT / 'results'
for d in [DATA_RAW, DATA_PROC, FIGURES, RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print(f'Running in Colab: {IN_COLAB}')
print(f'Project root: {ROOT}')


In [ ]:
# ── Cell 3: Global constants ──
KNOWN_HITS = {
    'Aplp1': {'fc': 2.34, 'log2_fc': 1.23, 'pvalue': 0.001, 'direction': 'up',   'analysis_type': 'parametric'},
    'Chl1':  {'fc': 1.87, 'log2_fc': 0.90, 'pvalue': 0.003, 'direction': 'up',   'analysis_type': 'parametric'},
    'Mapt':  {'fc': 1.56, 'log2_fc': 0.64, 'pvalue': 0.012, 'direction': 'up',   'analysis_type': 'nonparametric'},
    'App':   {'fc': 1.72, 'log2_fc': 0.78, 'pvalue': 0.008, 'direction': 'up',   'analysis_type': 'parametric'},
    'Aplp2': {'fc': 1.63, 'log2_fc': 0.70, 'pvalue': 0.015, 'direction': 'up',   'analysis_type': 'parametric'},
    'Sort1': {'fc': 1.45, 'log2_fc': 0.54, 'pvalue': 0.028, 'direction': 'up',   'analysis_type': 'parametric'},
    'Ctsd':  {'fc': 1.38, 'log2_fc': 0.46, 'pvalue': 0.034, 'direction': 'up',   'analysis_type': 'parametric'},
    'Sorl1': {'fc': 0.65, 'log2_fc':-0.62, 'pvalue': 0.022, 'direction': 'down', 'analysis_type': 'parametric'},
    'Lamp1': {'fc': 1.41, 'log2_fc': 0.50, 'pvalue': 0.031, 'direction': 'up',   'analysis_type': 'parametric'},
    'Ctsb':  {'fc': 1.33, 'log2_fc': 0.41, 'pvalue': 0.042, 'direction': 'up',   'analysis_type': 'parametric'},
}

ENDOSOMAL_PROTEINS = {
    'aplp1','aplp2','app','chl1','mapt','sorl1','sort1',
    'ctsd','ctsb','lamp1','lamp2','cadm1','cadm4','nrxn1',
    'nrcam','l1cam','ncam1','thy1','cntn1','nfasc','negr1',
    'opcml','ntm','sez6l','sez6','clstn1','clstn3','bsg','cd47','itm2b',
}

MODEL_COLORS = {
    'Simoes-3': '#3498db', 'Random-3': '#95a5a6',
    'Endosomal-Full': '#e67e22', 'Endosomal-Optimized': '#e74c3c',
    'Endosomal-XGB': '#2ecc71',
}
print('Constants defined.')


In [ ]:
# ── Cell 4: Data loading functions ──
def _find_header_row(df_raw, marker='sample id'):
    for i, row in df_raw.iterrows():
        if marker in str(row.iloc[0]).strip().lower():
            return i
    raise ValueError(f'Header row with marker {marker!r} not found')

def load_protein_list(path):
    df = pd.read_excel(path, sheet_name=0)
    df.columns = df.columns.str.strip()
    col_map = {}
    for c in df.columns:
        cl = c.lower()
        if 'accession' in cl: col_map[c] = 'accession'
        elif 'symbol' in cl or 'gene' in cl: col_map[c] = 'mouse_gene'
    df = df.rename(columns=col_map)
    return df[['accession','mouse_gene']].dropna(subset=['mouse_gene']).reset_index(drop=True)

def load_mouse_quantitative(s3_path):
    def parse_sheet(path, sheet):
        raw = pd.read_excel(path, sheet_name=sheet, header=None)
        hr = _find_header_row(raw)
        headers = raw.iloc[hr].tolist()
        data = raw.iloc[hr+1:].copy()
        data.columns = headers
        data = data.dropna(subset=[headers[0]])
        data = data.loc[:, data.columns.notna()]
        data = data.rename(columns={headers[0]: 'sample_id'})
        data['sample_id'] = data['sample_id'].astype(str).str.strip()
        data = data.replace({'undetected': np.nan, 'NaN': np.nan})
        for col in data.columns[1:]:
            data[col] = pd.to_numeric(data[col], errors='coerce')
        return data.reset_index(drop=True)
    
    fig2 = parse_sheet(s3_path, 'Data related to Fig. 2')
    fig2['sample_id'] = fig2['sample_id'].str.replace(' ', '_')
    fig2_rename = {'APLP2': 'Aplp2', 'CHL1': 'Chl1', 'APLP1': 'Aplp1', 'APP': 'App'}
    fig2 = fig2.rename(columns=fig2_rename)
    fig2_prots = [v for v in fig2_rename.values() if v in fig2.columns]
    
    figs1 = parse_sheet(s3_path, 'Data related to Fig. S1')
    figs1['bio'] = figs1['sample_id'].str.replace(r'_rep[l]?[i]?[c]?[a]?[t]?[e]?\s*\d+$','',regex=True).str.replace(' ','_').str.strip()
    figs1 = figs1.rename(columns={'n-APLP1':'Aplp1','n-CHL1':'Chl1','CADM4':'Cadm4','TUBB3':'Tubb3','TUBB2A':'Tubb2a'})
    s1_prots = [c for c in ['Aplp1','Chl1','Cadm4','Tubb3','Tubb2a'] if c in figs1.columns]
    figs1_avg = figs1.groupby('bio')[s1_prots].mean().reset_index().rename(columns={'bio':'sample_id'})
    
    merged = pd.merge(figs1_avg, fig2[['sample_id']+fig2_prots], on='sample_id', how='outer', suffixes=('_s1','_fig2'))
    for gene in ['Aplp1','Chl1']:
        c_s1 = f'{gene}_s1'; c_f2 = f'{gene}_fig2'
        if c_s1 in merged.columns and c_f2 in merged.columns:
            merged[gene] = merged[c_f2].combine_first(merged[c_s1])
            merged.drop(columns=[c_s1,c_f2], inplace=True)
    
    prot_cols = [c for c in merged.columns if c != 'sample_id']
    merged[prot_cols] = np.log2(merged[prot_cols].replace(0,np.nan))
    merged['label'] = merged['sample_id'].apply(lambda s: 0 if 'Control' in s else 1)
    return merged.set_index('sample_id')

def load_human_clinical(s3_path):
    raw7 = pd.read_excel(s3_path, sheet_name='Data related to Fig. 7', header=None)
    hr = None
    for i, row in raw7.iterrows():
        vals = [str(v).lower() for v in row if not pd.isna(v)]
        if any('control' in v and 'prodromal' in v for v in vals):
            hr = i; break
    data7 = raw7.iloc[hr+1:][[0,1,2]].copy()
    data7.columns = ['label','n_CHL1','n_APLP1']
    data7 = data7.dropna(subset=['label'])
    for c in data7.columns:
        data7[c] = pd.to_numeric(data7[c], errors='coerce')
    return data7.dropna(subset=['label','n_CHL1','n_APLP1']).reset_index(drop=True)

print('Data loading functions defined.')


In [ ]:
# ── Cell 5: Placeholder generator ──
def generate_placeholder_proteins(protein_list_df, n_significant=71, seed=42):
    rng = np.random.default_rng(seed)
    df = protein_list_df.copy()
    symbol_lower = df['mouse_gene'].str.lower()
    rows = []; used = set()
    
    for gene, stats in KNOWN_HITS.items():
        mask = symbol_lower == gene.lower()
        idx = df.index[mask]
        if len(idx) > 0:
            i = idx[0]; used.add(i)
            rows.append({'mouse_gene': df.loc[i,'mouse_gene'], 'accession': df.loc[i,'accession'], **stats, 'data_source':'placeholder'})
    
    remaining = [i for i in df.index if i not in used]
    n_extra = n_significant - len(rows)
    for i in rng.choice(remaining, size=min(n_extra,len(remaining)), replace=False):
        d = rng.choice(['up','down'], p=[0.65,0.35])
        fc = float(rng.uniform(1.3,2.5) if d=='up' else rng.uniform(0.4,0.75))
        rows.append({'mouse_gene':df.loc[i,'mouse_gene'],'accession':df.loc[i,'accession'],
                     'fc':fc,'log2_fc':np.log2(fc) if d=='up' else -np.log2(1/fc),
                     'pvalue':float(rng.uniform(0.001,0.049)),'direction':d,
                     'analysis_type':rng.choice(['parametric','nonparametric'],p=[0.73,0.27]),
                     'data_source':'placeholder'})
        used.add(i)
    
    for i in [j for j in df.index if j not in used]:
        fc = float(rng.lognormal(0,0.15))
        rows.append({'mouse_gene':df.loc[i,'mouse_gene'],'accession':df.loc[i,'accession'],
                     'fc':fc,'log2_fc':np.log2(fc),'pvalue':float(rng.uniform(0.05,1.0)),
                     'direction':'up' if fc>1 else 'down','analysis_type':'background','data_source':'placeholder'})
    return pd.DataFrame(rows).reset_index(drop=True)

def build_full_mouse_matrix(protein_list_df, quant_df, sig_df, seed=42):
    rng = np.random.default_rng(seed)
    samples = quant_df.index.tolist()
    labels = quant_df['label'].values
    real_proteins = [c for c in quant_df.columns if c != 'label']
    matrix = quant_df[real_proteins].copy()
    for gene in protein_list_df['mouse_gene'].tolist():
        if gene in matrix.columns: continue
        row = sig_df[sig_df['mouse_gene'].str.lower() == gene.lower()]
        if len(row) > 0 and row.iloc[0]['analysis_type'] in ('parametric','nonparametric'):
            log2_fc = float(row.iloc[0]['log2_fc'])
        else:
            log2_fc = 0.0
        matrix[gene] = [rng.normal(log2_fc*lbl, 0.3) for lbl in labels]
    matrix['label'] = labels
    matrix.index = samples
    return matrix

print('Placeholder functions defined.')


In [ ]:
# ── Cell 6: Load data ──
S1_PATH = DATA_RAW / 'aba6334_data_file_s1.xlsx'
S3_PATH = DATA_RAW / 'aba6334_data_file_s3.xlsx'

if S1_PATH.exists() and S3_PATH.exists():
    print('Loading real supplement data...')
    protein_list = load_protein_list(S1_PATH)
    quant_df = load_mouse_quantitative(S3_PATH)
    human_df = load_human_clinical(S3_PATH)
    sig_df = generate_placeholder_proteins(protein_list)
    mouse_matrix = build_full_mouse_matrix(protein_list, quant_df, sig_df)
    DATA_SOURCE = 'supplement'
else:
    print('Supplement files not found — generating placeholder data')
    genes = list(KNOWN_HITS.keys()) + [f'Protein_{i:04d}' for i in range(len(KNOWN_HITS), 1505)]
    protein_list = pd.DataFrame({'accession': [f'P{i:05d}' for i in range(1505)], 'mouse_gene': genes})
    sig_df = generate_placeholder_proteins(protein_list)
    rng = np.random.default_rng(RANDOM_STATE)
    samples = [f'Control_{i}' for i in range(1,5)] + [f'Vps35cKO_{i}' for i in range(1,5)]
    labels = [0]*4 + [1]*4
    data = {g: [rng.normal(float(sig_df.loc[sig_df.mouse_gene==g,'log2_fc'].values[0] if any(sig_df.mouse_gene==g) and sig_df.loc[sig_df.mouse_gene==g,'analysis_type'].values[0]!='background' else 0)*lbl, 0.3) for lbl in labels] for g in protein_list.mouse_gene}
    mouse_matrix = pd.DataFrame(data, index=samples)
    mouse_matrix['label'] = labels
    human_df = pd.DataFrame({'label':[0]*40+[1]*18,'n_APLP1':list(rng.normal(4.8,0.9,40))+list(rng.normal(10.5,2.5,18)),'n_CHL1':list(rng.normal(0.07,0.02,40))+list(rng.normal(0.16,0.05,18))})
    DATA_SOURCE = 'placeholder'

sig_filt = sig_df[sig_df.analysis_type.isin(['parametric','nonparametric'])]
print(f'\nData source: {DATA_SOURCE}')
print(f'Proteins: {len(protein_list)} | Significant: {len(sig_filt)} | Mouse samples: {len(mouse_matrix)} | Human subjects: {len(human_df)}')
for gene in ['Aplp1','Chl1','Mapt']:
    m = sig_filt[sig_filt.mouse_gene.str.lower()==gene.lower()]
    print(f'  {"✓" if len(m) else "✗"} {gene}' + (f': FC={m.iloc[0].fc:.2f}' if len(m) else ' NOT FOUND'))


In [ ]:
# ── Cell 7: Preprocessing ──
def preprocess_matrix(df):
    label_col = df['label'].copy() if 'label' in df.columns else None
    prot = df.drop(columns=['label'], errors='ignore').replace(0, np.nan)
    flat = prot.values.ravel(); flat = flat[np.isfinite(flat)]
    if flat.max() >= 50:
        prot = np.log2(prot.replace(0,np.nan))
    missing_frac = prot.isna().mean(axis=0)
    prot = prot.loc[:, missing_frac <= 0.5]
    from sklearn.impute import KNNImputer
    imputer = KNNImputer(n_neighbors=5, weights='distance')
    prot_imp = pd.DataFrame(imputer.fit_transform(prot.values), index=prot.index, columns=prot.columns)
    prot_imp = prot_imp.subtract(prot_imp.median(axis=1), axis=0)
    if label_col is not None:
        prot_imp['label'] = label_col.values
    return prot_imp

matrix = preprocess_matrix(mouse_matrix)
prot_cols = [c for c in matrix.columns if c != 'label']
X = matrix[prot_cols].values.astype(float)
y = matrix['label'].values.astype(int)
print(f'Preprocessed matrix: {X.shape}, classes: {np.bincount(y)}')


In [ ]:
# ── Cell 8: Model definitions ──
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
import xgboost as xgb

def get_models():
    return {
        'Simoes-3': Pipeline([('sc',StandardScaler()),('clf',LogisticRegression(penalty='l2',C=1.0,solver='lbfgs',max_iter=5000,random_state=42,class_weight='balanced'))]),
        'Random-3': Pipeline([('sc',StandardScaler()),('clf',LogisticRegression(penalty='l2',C=1.0,solver='lbfgs',max_iter=5000,random_state=42,class_weight='balanced'))]),
        'Endosomal-Full': Pipeline([('sc',StandardScaler()),('clf',LogisticRegression(penalty='elasticnet',solver='saga',l1_ratio=0.5,C=0.1,max_iter=10000,random_state=42,class_weight='balanced'))]),
        'Endosomal-Optimized': Pipeline([('sc',StandardScaler()),('clf',LogisticRegression(penalty='l2',C=1.0,solver='lbfgs',max_iter=5000,random_state=42,class_weight='balanced'))]),
        'Endosomal-XGB': Pipeline([('sc',StandardScaler()),('clf',xgb.XGBClassifier(n_estimators=200,max_depth=3,learning_rate=0.05,subsample=0.8,colsample_bytree=0.7,min_child_weight=5,reg_alpha=1.0,reg_lambda=5.0,eval_metric='logloss',objective='binary:logistic',random_state=42,verbosity=0))]),
    }

endo_features = [f for f in prot_cols if f.lower() in ENDOSOMAL_PROTEINS]
simoes3 = [f for f in prot_cols if f.lower() in {'aplp1','chl1','mapt'}]
rng0 = np.random.default_rng(42)
rand3 = list(rng0.choice([f for f in prot_cols if f.lower() not in ENDOSOMAL_PROTEINS], size=3, replace=False))
feature_sets = {'Simoes-3':simoes3,'Random-3':rand3,'Endosomal-Full':endo_features,'Endosomal-Optimized':endo_features,'Endosomal-XGB':endo_features}
print(f'Endosomal features: {len(endo_features)}')
print(f'Simoes-3 present: {simoes3}')
print(f'Random-3: {rand3}')


In [ ]:
# ── Cell 9: Stability selection ──
def stability_selection(X, y, n_iterations=200, sample_fraction=0.5, seed=42):
    rng = np.random.default_rng(seed)
    C_range = np.logspace(-3,1,20)
    n_samples, n_features = X.shape
    counts = np.zeros(n_features)
    sub_size = max(2, int(n_samples * sample_fraction))
    for i in range(n_iterations):
        idx = rng.choice(n_samples, size=sub_size, replace=False)
        Xs, ys = X[idx], y[idx]
        if len(np.unique(ys)) < 2: continue
        try:
            m = LogisticRegression(penalty='l1',C=float(rng.choice(C_range)),solver='saga',max_iter=5000,random_state=int(rng.integers(1e6)),class_weight='balanced')
            m.fit(Xs,ys)
            if m.coef_ is not None and not np.any(np.isnan(m.coef_)):
                counts += np.abs(m.coef_[0]) > 1e-8
        except: pass
    return counts / n_iterations

print('Stability selection function defined.')


In [ ]:
# ── Cell 10: Cross-validation ──
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
models_def = get_models()
results = {n: {'aucs':[],'all_y_true':[],'all_y_prob':[]} for n in models_def}

print('Running 5-fold × 3-repeat CV...')
splits = list(cv.split(X, y))
for fold_i, (tr_idx, val_idx) in enumerate(splits):
    for name, base_pipe in models_def.items():
        feat_names = feature_sets[name]
        feat_idx = [prot_cols.index(f) for f in feat_names if f in prot_cols]
        if not feat_idx: continue
        Xs = X[:, feat_idx]
        Xtr, Xval, ytr, yval = Xs[tr_idx], Xs[val_idx], y[tr_idx], y[val_idx]
        if name in ('Endosomal-Optimized','Endosomal-XGB'):
            from sklearn.preprocessing import StandardScaler as SS
            Xs_sc = SS().fit_transform(Xtr)
            probs = stability_selection(Xs_sc, ytr, n_iterations=50, sample_fraction=0.5, seed=42+fold_i)
            mask = probs >= 0.6
            if mask.sum() < 2:
                top3 = np.argsort(probs)[-3:]; mask = np.zeros(len(probs),bool); mask[top3]=True
            Xtr, Xval = Xtr[:,mask], Xval[:,mask]
        if len(np.unique(yval)) < 2: continue
        pipe = clone(base_pipe)
        try:
            pipe.fit(Xtr, ytr)
            yp = pipe.predict_proba(Xval)[:,1]
            results[name]['aucs'].append(roc_auc_score(yval, yp))
            results[name]['all_y_true'].extend(yval.tolist())
            results[name]['all_y_prob'].extend(yp.tolist())
        except: pass

for name, res in results.items():
    if res['aucs']:
        aucs = res['aucs']
        res.update({'mean_auc':np.mean(aucs),'std_auc':np.std(aucs),'ci_95_low':np.percentile(aucs,2.5),'ci_95_high':np.percentile(aucs,97.5),'n_folds':len(aucs)})
        print(f'  {name:26s}: AUC={np.mean(aucs):.3f} ± {np.std(aucs):.3f} (95%CI: {np.percentile(aucs,2.5):.3f}–{np.percentile(aucs,97.5):.3f})')


In [ ]:
# ── Cell 11: ROC plot ──
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
fig, ax = plt.subplots(figsize=(7,6))
for name, res in results.items():
    if not res.get('all_y_true',[]): continue
    fpr, tpr, _ = roc_curve(res['all_y_true'], res['all_y_prob'])
    ax.plot(fpr, tpr, color=MODEL_COLORS.get(name,'gray'), linewidth=2,
            label=f'{name} (AUC={res["mean_auc"]:.3f} ± {res["std_auc"]:.3f})')
ax.plot([0,1],[0,1],'k--',lw=0.5,alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'Model Comparison [{DATA_SOURCE}]')
ax.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.savefig(FIGURES / 'roc_comparison.png', dpi=150)
plt.show()
print('ROC plot saved.')


In [ ]:
# ── Cell 12: Volcano plot ──
sig_all = sig_df.copy()
sig_all['-log10_p'] = -np.log10(sig_all['pvalue'].clip(1e-10))
val3 = ['aplp1','chl1','mapt']
colors = np.where(
    sig_all.mouse_gene.str.lower().isin(val3), '#f1c40f',
    np.where((sig_all.analysis_type!='background')&(sig_all.direction=='up'), '#e74c3c',
    np.where((sig_all.analysis_type!='background')&(sig_all.direction=='down'), '#3498db', '#cccccc')))
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(sig_all.log2_fc, sig_all['-log10_p'], c=colors, s=12, alpha=0.7, linewidths=0)
ax.axhline(-np.log10(0.05),ls='--',lw=0.8,color='gray',alpha=0.6)
for _, r in sig_all[sig_all.mouse_gene.str.lower().isin(val3)].iterrows():
    ax.annotate(r.mouse_gene,(r.log2_fc,r['-log10_p']),xytext=(5,3),textcoords='offset points',fontsize=8,fontweight='bold')
patches = [mpatches.Patch(color='#e74c3c',label='Sig ↑'),mpatches.Patch(color='#3498db',label='Sig ↓'),
           mpatches.Patch(color='#f1c40f',label='Simoes validated'),mpatches.Patch(color='#cccccc',label='Background')]
ax.legend(handles=patches,loc='upper left')
ax.set_xlabel('log₂ FC (VPS35-cKO / Control)'); ax.set_ylabel('-log₁₀ p')
ax.set_title('CSF Proteomics: VPS35-cKO vs Control')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES / 'volcano.png', dpi=150)
plt.show()


In [ ]:
# ── Cell 13: Stability selection analysis ──
endo_idx = [prot_cols.index(f) for f in endo_features if f in prot_cols]
X_endo = X[:, endo_idx]
from sklearn.preprocessing import StandardScaler
X_sc = StandardScaler().fit_transform(X_endo)
print('Running stability selection (200 iterations)...')
sel_probs = stability_selection(X_sc, y, n_iterations=200, sample_fraction=0.5)
stable = [(f, p) for f, p in sorted(zip(endo_features, sel_probs), key=lambda x: -x[1]) if p >= 0.6]
print(f'Stable features (prob ≥ 0.60): {[f for f,_ in stable]}')
import pandas as pd
stab_df = pd.DataFrame({'protein':endo_features,'selection_frequency':sel_probs}).sort_values('selection_frequency',ascending=False)
stab_df.to_csv(RESULTS / 'stability_selection_results.csv', index=False)

fig, ax = plt.subplots(figsize=(7, min(len(endo_features),30)*0.3+1))
top30 = stab_df.head(30).iloc[::-1]
colors_s = ['#e74c3c' if p>=0.6 else '#cccccc' for p in top30.selection_frequency]
ax.barh(top30.protein, top30.selection_frequency, color=colors_s, edgecolor='none')
ax.axvline(0.6,ls='--',lw=1.2,color='black',alpha=0.7,label='Threshold=0.60')
ax.set_xlabel('Selection Probability'); ax.set_title('Stability Selection')
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES / 'stability_selection.png', dpi=150)
plt.show()


In [ ]:
# ── Cell 14: SHAP analysis ──
import shap
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Train XGBoost on endosomal features
X_tr, X_te, y_tr, y_te = train_test_split(X_endo, y, test_size=0.3, random_state=42, stratify=y)
xgb_pipe = get_models()['Endosomal-XGB']
xgb_pipe.fit(X_tr, y_tr)

# Extract clf and scale
scaler_ = xgb_pipe.named_steps['sc']
clf_ = xgb_pipe.named_steps['clf']
X_tr_sc = scaler_.transform(X_tr)
X_te_sc = scaler_.transform(X_te)

explainer = shap.TreeExplainer(clf_, data=X_tr_sc, feature_perturbation='interventional')
shap_vals = explainer.shap_values(X_te_sc)
if isinstance(shap_vals, list): shap_vals = shap_vals[1]

mean_abs = np.abs(shap_vals).mean(axis=0)
imp_df = pd.DataFrame({'feature':endo_features,'mean_abs_shap':mean_abs}).sort_values('mean_abs_shap',ascending=False)
print('Top 10 features by mean |SHAP|:')
print(imp_df.head(10).to_string(index=False))
imp_df.to_csv(RESULTS / 'shap_importance.csv', index=False)


In [ ]:
# ── Cell 15: SHAP bar chart ──
top = imp_df.head(20).iloc[::-1]
colors_sh = ['#e74c3c' if i>=17 else '#aaaaaa' for i in range(20)][::-1]
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(top.feature, top.mean_abs_shap, color=colors_sh, edgecolor='none')
ax.set_xlabel('Mean |SHAP value|'); ax.set_title('Feature Importance (SHAP) — Endosomal-XGB')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES / 'shap_bar.png', dpi=150)
plt.show()


In [ ]:
# ── Cell 16: Human clinical data analysis ──
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

print(f'Human clinical data: {len(human_df)} subjects')
print(human_df['label'].value_counts())
feat_h = [c for c in ['n_APLP1','n_CHL1','tau_abeta42'] if c in human_df.columns]
X_h = human_df[feat_h].dropna().values
y_h = human_df.loc[human_df[feat_h].notna().all(axis=1),'label'].values
pipe_h = Pipeline([('sc',StandardScaler()),('clf',LogisticRegression(penalty='l2',C=1.0,solver='lbfgs',max_iter=5000,class_weight='balanced',random_state=42))])
aucs_h = cross_val_score(pipe_h, X_h, y_h, cv=5, scoring='roc_auc')
print(f'\nReal human AUC (features: {feat_h}):\n  {aucs_h.mean():.3f} ± {aucs_h.std():.3f}')

fig, ax = plt.subplots(figsize=(6,5))
for lbl, color, name in [(0,'#3498db','Control'),(1,'#e74c3c','Prodromal AD')]:
    sub = human_df[human_df.label==lbl]
    ax.scatter(sub['n_APLP1'], sub['n_CHL1'], color=color, label=name, alpha=0.8, s=40)
ax.set_xlabel('n-APLP1'); ax.set_ylabel('n-CHL1'); ax.set_title('Human CSF: APLP1 vs CHL1')
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES / 'human_biomarkers.png', dpi=150)
plt.show()


In [ ]:
# ── Cell 17: Synthetic human cohort ──
def generate_synthetic_human(sig_df, n_ctrl=250, n_mci=200, n_ad=150, noise=0.3, seed=42):
    rng = np.random.default_rng(seed)
    labels = [0]*n_ctrl + [1]*n_mci + [2]*n_ad
    n_total = len(labels)
    data = {}
    for _, row in sig_df.iterrows():
        g = row['mouse_gene']
        fc = float(row.get('log2_fc',0.0))
        is_sig = row.get('analysis_type','background') in ('parametric','nonparametric')
        vals = []
        for lbl in labels:
            mean = (fc*lbl/2 if lbl==1 else fc if lbl==2 else 0.0) if is_sig else 0.0
            vals.append(rng.normal(mean, noise*(1+lbl*0.1)))
        data[g] = vals
    df = pd.DataFrame(data)
    df.insert(0,'sample_id',[f'SYN_{i:04d}' for i in range(1,n_total+1)])
    df.insert(1,'diagnosis',labels)
    return df

print('Generating 600-subject synthetic cohort...')
synth = generate_synthetic_human(sig_filt)
print(f'Shape: {synth.shape}')
print(synth['diagnosis'].value_counts().sort_index())
synth.to_csv(DATA_PROC / 'synthetic_human_data.csv', index=False)
print('Saved synthetic_human_data.csv')


In [ ]:
# ── Cell 18: Final summary ──
import json
summary = {
    'data_source': DATA_SOURCE,
    'n_proteins': len(protein_list),
    'n_significant': len(sig_filt),
    'n_mouse_samples': len(mouse_matrix),
    'n_human_subjects': len(human_df),
    'model_aucs': {n: f"{res.get('mean_auc',0):.3f} ± {res.get('std_auc',0):.3f}" for n,res in results.items() if res.get('aucs')},
    'figures_generated': [str(p.name) for p in FIGURES.glob('*.png')],
    'outputs': [str(p.name) for p in list(DATA_PROC.glob('*.csv')) + list(RESULTS.glob('*.csv'))],
}
print(json.dumps(summary, indent=2))
with open(RESULTS / 'pipeline_summary.json','w') as f:
    json.dump(summary, f, indent=2)
print('\nPipeline complete! All outputs saved.')
